# 01 · Data collection

**Goal:** download official air quality measurements for Milan from the Regione Lombardia open data portal and check what we got.

**Source:** ARPA Lombardia (the regional environmental protection agency) publishes every measurement from its monitoring network on [dati.lombardia.it](https://www.dati.lombardia.it), through a public Socrata API (no key needed).

| Dataset ID | Name | Used for |
|---|---|---|
| [`ib47-atvt`](https://www.dati.lombardia.it/d/ib47-atvt) | Stazioni qualità dell'aria | station and sensor metadata |
| [`nr8w-tj77`](https://www.dati.lombardia.it/d/nr8w-tj77) | Dati sensori aria 2010–2017 | measurements 2015–2017 |
| [`g2hp-ar79`](https://www.dati.lombardia.it/d/g2hp-ar79) | Dati sensori aria dal 2018 | measurements 2018–2025 |
| [`nicp-bhqi`](https://www.dati.lombardia.it/d/nicp-bhqi) | Dati sensori aria | current year (2026, provisional) |

ARPA moves each year into an archive once it is fully validated, so we stitch three datasets together. Each one is queried only for its own date window, so no timestamp is downloaded twice.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import pandas as pd
from src.config import DATA_RAW, SENSORS
from src.fetch import download_all, load_raw

## Download (or reuse the local copy)

Raw data is not committed to git (it is ~25 MB and anyone can re-download it). If the files are missing, this cell downloads them; you can also run `python scripts/download_data.py`.

In [2]:
stations_path = DATA_RAW / "stations_milano.csv"
meas_path = DATA_RAW / "measurements_milano.csv"
if not meas_path.exists():
    download_all()
stations = pd.read_csv(stations_path, dtype=str)
raw = load_raw(meas_path)  # keeps ARPA's flag "NA" as text (see notebook 02)
print(f"{len(raw):,} measurement rows, {raw['idsensore'].nunique()} sensors")
raw.head()

537,866 measurement rows, 10 sensors


,idsensore,data,valore,stato,idoperatore,dataset
0,10273,2015-01-01T00:00:00.000,65.0,VA,1,nr8w-tj77
1,10273,2015-01-02T00:00:00.000,69.0,VA,1,nr8w-tj77
2,10273,2015-01-03T00:00:00.000,84.0,VA,1,nr8w-tj77
3,10273,2015-01-04T00:00:00.000,40.0,VA,1,nr8w-tj77
4,10273,2015-01-05T00:00:00.000,27.0,VA,1,nr8w-tj77


## The Milan sensors we use

Milan has more sensors than this, but we keep only those that measure PM10, PM2.5 or NO2 **and were active for the whole 2015–2026 period**. If stations open or close mid-way, a city average changes just because the set of stations changed, which would look like a fake trend.

In [3]:
chosen = stations[stations["idsensore"].isin(SENSORS)].copy()
chosen["pollutant"] = chosen["idsensore"].map(lambda s: SENSORS[s][0])
chosen[["pollutant", "idsensore", "nomestazione", "idstazione", "unitamisura", "datastart", "storico", "lat", "lng"]].sort_values(["pollutant", "nomestazione"])

,pollutant,idsensore,nomestazione,idstazione,unitamisura,datastart,storico,lat,lng
3,NO2,10279,Milano Pascal Città Studi,705,µg/m³,2007-06-16T00:00:00.000,N,45.47834700,9.23166700
35,NO2,5531,Milano Verziere,528,µg/m³,1988-11-24T00:00:00.000,N,45.46334886,9.19532511
36,NO2,5542,Milano v.Liguria,539,µg/m³,1991-10-24T00:00:00.000,N,45.44386047,9.16794489
32,NO2,5504,Milano v.Marche,501,µg/m³,1980-09-18T00:00:00.000,N,45.49631883,9.19093444
38,NO2,5551,Milano v.Senato,548,µg/m³,1995-04-21T00:00:00.000,N,45.47050100,9.19746075
1,PM10,10273,Milano Pascal Città Studi,705,µg/m³,2007-04-21T00:00:00.000,N,45.47834700,9.23166700
69,PM10,6956,Milano Verziere,528,µg/m³,2001-11-01T00:00:00.000,N,45.46334886,9.19532511
7,PM10,10320,Milano v.Senato,548,µg/m³,2007-08-10T00:00:00.000,N,45.47050100,9.19746075
6,PM2.5,10283,Milano Pascal Città Studi,705,µg/m³,2007-03-29T00:00:00.000,N,45.47834700,9.23166700
18,PM2.5,17122,Milano v.Senato,548,µg/m³,2012-09-04T00:00:00.000,N,45.47050100,9.19746075


Excluded on purpose:
- **Milano v.Marche PM10 / PM2.5**: PM10 only starts in 2018 (and has no 2025 data), PM2.5 starts in 2021. Its NO2 sensor runs the whole period, so we keep that one.
- **Zavattari, Abbiategrasso, Parco Lambro**: closed in July 2018 (`storico = S`).

## What does the raw data look like?

In [4]:
raw["data"] = pd.to_datetime(raw["data"])
raw["year"] = raw["data"].dt.year
raw["pollutant"] = raw["idsensore"].map(lambda s: SENSORS[s][0])
summary = raw.groupby(["pollutant", "idsensore"]).agg(
    first=("data", "min"), last=("data", "max"), rows=("valore", "size"),
    share_flagged_invalid=("stato", lambda s: round((s != "VA").mean() * 100, 2)),
)
summary

first                last    rows  \
pollutant idsensore                                          
NO2       10279     2015-01-01 2026-11-17 13:00:00  103643   
          5504      2015-01-01 2026-11-17 16:00:00  103956   
          5531      2015-01-01 2026-11-17 16:00:00  102869   
          5542      2015-01-01 2026-11-17 16:00:00  103010   
          5551      2015-01-01 2026-11-17 16:00:00  103247   
PM10      10273     2015-01-01 2026-11-16 00:00:00    4197   
          10320     2015-01-01 2026-11-16 00:00:00    4258   
          6956      2015-01-01 2026-11-16 00:00:00    4261   
PM2.5     10283     2015-01-01 2026-11-16 00:00:00    4180   
          17122     2015-01-01 2026-11-16 00:00:00    4245   

                     share_flagged_invalid  
pollutant idsensore                         
NO2       10279                       4.72  
          5504                        3.71  
          5531                        2.10  
          5542                        3.91  
          5551                        2.99  
PM10      10273                       2.00  
          10320                       2.87  
          6956                        3.43  
PM2.5     10283                       1.99  
          17122                       3.20

Two things to notice:

1. **PM10 and PM2.5 have about one row per day; NO2 has about 24.** Particulate matter is measured as a 24-hour average (filters collect particles over a day), while NO2 analysers report every hour. So hour-of-day patterns are only possible for NO2.
2. **The `stato` column is ARPA's validity flag:** `VA` = valid, `NA` = not valid. A small share of rows are flagged invalid; notebook 02 handles them.

In [5]:
rows_per_year = raw.pivot_table(index="year", columns="pollutant", values="valore", aggfunc="size")
rows_per_year

pollutant,NO2,PM10,PM2.5
year,,,
2015,43418,1063,711
2016,43870,1071,697
2017,43698,1070,703
2018,43482,1067,702
2019,43634,1065,710
2020,43703,1083,724
2021,43008,1056,700
2022,42951,1058,690
2023,43466,1088,721


**Takeaway:** the API gives continuous coverage from 2015 to today for all 10 sensors. 2026 is the current, incomplete year, and its values are provisional until ARPA's final validation.